# Madrid2Dam · h4d2 live dashboard

Upload a test parquet (same format as the organiser files), pick the city and label budgets, and press **Run h4d2**.
The real h4d2 model runs here on the hub and the dashboard below fills in with your results.

* Amsterdam: one prediction per label budget n = 5, 25, 50, 100, 200 (the committed Stage 2 models, seed 42).
* Madrid: the Stage 1 Madrid model.
* If the file still has *weighted_mean_year*, the dashboard also scores the predictions (macro F1, confusion matrix, right/wrong map).
* The provided city file is used only as unlabelled spatial context, exactly as in the submission.

Run the two cells below from the top (Run ▶▶). First run takes about a minute to load the models.

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser('~/Hack4dev-NCRST'))
os.environ.setdefault('H4D_DATA_DIR', os.path.expanduser('~/Building_Age_Classification/data'))
from pathlib import Path
import ipywidgets as W
from IPython.display import display, clear_output
import h4d2.dashboard_live as DL

up = W.FileUpload(accept='.parquet', multiple=False, description='Upload parquet')
path = W.Text(placeholder='…or a path on the hub, e.g. ~/dryrun/amsterdam_test.parquet', layout=W.Layout(width='520px'))
city = W.ToggleButtons(options=['Amsterdam', 'Madrid'], value='Amsterdam')
ns = W.SelectMultiple(options=[5, 25, 50, 100, 200], value=(5, 25, 50, 100, 200), rows=5, description='n per class')
ctx = W.Checkbox(value=True, description='Use the provided city file as unlabelled neighbours')
go = W.Button(description='Run h4d2', button_style='primary', icon='play')
log = W.Output(); dash = W.Output()

def on_run(_):
    log.clear_output(); dash.clear_output()
    with log:
        if up.value:
            f = up.value[0]
            dest = DL.RUNS / f['name']; dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(bytes(f['content'])); src = dest
        elif path.value.strip():
            src = Path(os.path.expanduser(path.value.strip()))
        else:
            print('Upload a parquet or give a path first.'); return
        print('Running h4d2 on', src.name, '…')
        ds = DL.run(src, city.value, ns=sorted(ns.value), use_context=ctx.value)
    with dash:
        DL.show(ds, note=f'{src.name} · {city.value} · ' + ('n = ' + ', '.join(map(str, sorted(ns.value))) if city.value == 'Amsterdam' else 'Madrid Stage 1 model'))

go.on_click(on_run)
display(W.VBox([W.HTML('<b>1. Data</b>'), W.HBox([up, path]), W.HTML('<b>2. Settings</b>'), city, ns, ctx, go, log, dash]))

Just want the results dashboard without running anything? Run the cell below.

In [ ]:
DL.show()